# Strategy comparison

**Purpose**: Visual comparison of cold-start / query strategies: final AP, training time, query efficiency, and multi-metric trade-offs.

## Prerequisites
- `experiments/configs/cold_start_config.yaml` exists (or update `config_path`).
- If using real results: JSON logs in `../results/` following your experiment schema.

## How to run
- Run top-to-bottom.
- Replace the simulated section with `pd.read_json`/custom loader for real results.

## Notes
- Paths in this notebook are repository- and machine-specific; update them before running.
- If you're running from the `notebooks/` folder, the `PROJECT_ROOT` logic should work as-is.


## Strategy Comparison Notebook
Compare cold-start strategies using either real experiment logs or (in this notebook) simulated metrics.

**Tip**: replace the simulated `performance_data` with loaded results from your `results/` folder.

## Data source
This notebook currently **simulates** results so the plots run out-of-the-box.
To use real experiments, load your JSON logs into a DataFrame with columns like:
`strategy`, `final_ap`, `best_ap`, `training_time_min`, `std_dev`.


In [ ]:
import sys
sys.path.append('..')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import json
from pathlib import Path
import yaml

# Set plotting style
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")

# Load configuration
config_path = Path("../experiments/configs/cold_start_config.yaml")
with open(config_path, 'r') as f:
    config = yaml.safe_load(f)

print("Configuration loaded:")
print(f"Dataset: {config['base']['dataset']}")
print(f"Query Strategy: {config['base']['query_strategy']}")
print(f"Cold Start Strategies: {list(config['cold_start_strategies'].keys())}")

# Create sample data for demonstration
# In practice, you would load actual experiment results
strategies = list(config['cold_start_strategies'].keys())
n_strategies = len(strategies)

# Simulate performance metrics
np.random.seed(42)
performance_data = {
    'strategy': strategies,
    'final_ap': np.random.uniform(0.3, 0.7, n_strategies),
    'best_ap': np.random.uniform(0.35, 0.75, n_strategies),
    'training_time_min': np.random.uniform(30, 120, n_strategies),
    'query_efficiency': np.random.uniform(0.01, 0.05, n_strategies),
    'std_dev': np.random.uniform(0.02, 0.08, n_strategies)
}

df = pd.DataFrame(performance_data)
df = df.sort_values('final_ap', ascending=False)

print("\nSample Performance Data:")
print(df.to_string())

## Plot 1: Bar chart of final AP

In [ ]:
plt.figure(figsize=(10, 6))
bars = plt.bar(range(len(df)), df['final_ap'], 
               yerr=df['std_dev'], capsize=5,
               color=sns.color_palette("viridis", len(df)))

plt.xlabel('Strategy')
plt.ylabel('Final AP@[IoU=0.50:0.95]')
plt.title('Cold Start Strategy Comparison')
plt.xticks(range(len(df)), df['strategy'], rotation=45, ha='right')
plt.ylim(0, 1.0)

# Add value labels
for i, (idx, row) in enumerate(df.iterrows()):
    plt.text(i, row['final_ap'] + 0.02, f"{row['final_ap']:.3f}", 
             ha='center', va='bottom', fontsize=10)

plt.tight_layout()
plt.show()

## Plot 2: Scatter plot of AP vs Training Time

In [ ]:
plt.figure(figsize=(10, 6))
scatter = plt.scatter(df['training_time_min'], df['final_ap'], 
                     s=200, c=range(len(df)), cmap='viridis', alpha=0.7)

# Add labels for each point
for i, (idx, row) in enumerate(df.iterrows()):
    plt.annotate(row['strategy'], 
                (row['training_time_min'], row['final_ap']),
                xytext=(5, 5), textcoords='offset points',
                fontsize=9)

plt.xlabel('Training Time (minutes)')
plt.ylabel('Final AP')
plt.title('Performance vs Training Time Trade-off')
plt.colorbar(scatter, label='Strategy Rank')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## Plot 3: Radar chart for multi-dimensional comparison

In [ ]:
from math import pi

# Normalize metrics for radar chart
metrics = ['final_ap', 'best_ap', 'query_efficiency']
df_normalized = df.copy()
for metric in metrics:
    df_normalized[metric] = (df[metric] - df[metric].min()) / (df[metric].max() - df[metric].min())

# Number of variables
categories = metrics
N = len(categories)

# Create radar chart
fig, ax = plt.subplots(figsize=(10, 10), subplot_kw=dict(projection='polar'))

# Draw one axe per variable + add labels
angles = [n / float(N) * 2 * pi for n in range(N)]
angles += angles[:1]

# Draw polygon for each strategy
for idx, row in df_normalized.iterrows():
    values = row[metrics].values.flatten().tolist()
    values += values[:1]
    ax.plot(angles, values, linewidth=2, linestyle='solid', label=row['strategy'])
    ax.fill(angles, values, alpha=0.1)

# Draw axis lines
ax.set_xticks(angles[:-1])
ax.set_xticklabels(categories)

# Draw ylabels
ax.set_rlabel_position(0)
plt.yticks([0.2, 0.4, 0.6, 0.8], ["0.2", "0.4", "0.6", "0.8"], color="grey", size=7)
plt.ylim(0, 1)

# Add legend
plt.legend(loc='upper right', bbox_to_anchor=(0.1, 0.1))

plt.title('Multi-dimensional Strategy Comparison', size=15, y=1.1)
plt.tight_layout()
plt.show()

## Plot 4: Learning curve comparison (simulated)

In [ ]:
plt.figure(figsize=(12, 8))

# Generate simulated learning curves
epochs = range(1, 21)
for i, strategy in enumerate(strategies):
    # Simulate different learning curves
    if strategy == 'random':
        curve = np.linspace(0.1, 0.4, len(epochs)) + np.random.normal(0, 0.02, len(epochs))
    elif strategy == 'diversity':
        curve = np.linspace(0.2, 0.6, len(epochs)) + np.random.normal(0, 0.02, len(epochs))
    elif strategy == 'weak_supervision':
        curve = np.linspace(0.15, 0.55, len(epochs)) + np.random.normal(0, 0.02, len(epochs))
    else:
        curve = np.linspace(0.1, 0.5, len(epochs)) + np.random.normal(0, 0.02, len(epochs))
    
    plt.plot(epochs, curve, '-o', label=strategy, markersize=4, linewidth=2)

plt.xlabel('Epoch')
plt.ylabel('AP@[IoU=0.50:0.95]')
plt.title('Simulated Learning Curves Comparison')
plt.grid(True, alpha=0.3)
plt.legend()
plt.tight_layout()
plt.show()

## Plot 5: Query efficiency over AL cycles

In [ ]:
plt.figure(figsize=(12, 6))

# Simulate query efficiency
cycles = range(1, 6)
for i, strategy in enumerate(strategies[:3]):  # Show top 3
    # Different efficiency patterns
    if strategy == 'diversity':
        efficiency = [0.02, 0.015, 0.01, 0.008, 0.006]
    elif strategy == 'weak_supervision':
        efficiency = [0.015, 0.012, 0.01, 0.009, 0.008]
    else:
        efficiency = [0.01, 0.008, 0.007, 0.006, 0.005]
    
    plt.plot(cycles, efficiency, '-s', label=strategy, markersize=8, linewidth=2)

plt.xlabel('Active Learning Cycle')
plt.ylabel('AP Gain per Query')
plt.title('Query Efficiency over AL Cycles')
plt.grid(True, alpha=0.3)
plt.legend()
plt.tight_layout()
plt.show()

## Create summary table

In [ ]:
print("\n" + "="*60)
print("STRATEGY COMPARISON SUMMARY")
print("="*60)

summary_table = df[['strategy', 'final_ap', 'best_ap', 'training_time_min', 'query_efficiency']].copy()
summary_table['final_ap'] = summary_table['final_ap'].apply(lambda x: f"{x:.3f}")
summary_table['best_ap'] = summary_table['best_ap'].apply(lambda x: f"{x:.3f}")
summary_table['training_time_min'] = summary_table['training_time_min'].apply(lambda x: f"{x:.1f}")
summary_table['query_efficiency'] = summary_table['query_efficiency'].apply(lambda x: f"{x:.4f}")

print(summary_table.to_string(index=False))

## Recommendations based on analysis

In [ ]:
print("\n" + "="*60)
print("RECOMMENDATIONS")
print("="*60)

best_strategy = df.iloc[0]
print(f"1. Best Overall Strategy: {best_strategy['strategy']}")
print(f"   - Final AP: {best_strategy['final_ap']:.3f}")
print(f"   - Training Time: {best_strategy['training_time_min']:.1f} min")
print(f"   - Query Efficiency: {best_strategy['query_efficiency']:.4f}")

# Find best trade-off (AP per minute)
df['ap_per_minute'] = df['final_ap'] / df['training_time_min']
best_tradeoff = df.loc[df['ap_per_minute'].idxmax()]
print(f"\n2. Best Time-Efficiency Trade-off: {best_tradeoff['strategy']}")
print(f"   - AP per minute: {best_tradeoff['ap_per_minute']:.4f}")

print("\n3. Strategy Selection Guidelines:")
print("   - For maximum performance: Choose diversity-based methods")
print("   - For limited time: Choose simple_diversity or random")
print("   - For data efficiency: Choose weak_supervision or self_supervised")

## Save analysis results

In [ ]:
output_dir = Path("../results/analysis")
output_dir.mkdir(parents=True, exist_ok=True)

# Save DataFrame
df.to_csv(output_dir / "strategy_comparison.csv", index=False)

# Save plots
fig1 = plt.figure(figsize=(10, 6))
bars = plt.bar(range(len(df)), df['final_ap'], yerr=df['std_dev'], capsize=5)
plt.xticks(range(len(df)), df['strategy'], rotation=45, ha='right')
plt.ylabel('Final AP')
plt.title('Strategy Comparison')
plt.tight_layout()
plt.savefig(output_dir / "strategy_comparison.png", dpi=300, bbox_inches='tight')

print(f"\nAnalysis results saved to: {output_dir}")